In [1]:
import os
import json
import re
import pdfplumber
from PIL import Image
from pypdf import PdfReader, PdfWriter
# from pipe_fn import pipe
from transformers import pipeline
from output_utils import save_split_output
from confidence_utils import (
    calculate_eob_confidence,
    _unwrap_vlm_output,calculate_model_confidence
)
# =========================================================
# LOAD MODEL
# =========================================================

pipe = pipeline(
    "image-text-to-text",
    model="Qwen/Qwen3-VL-2B-Instruct",
    device_map="auto"
)

COLUMN_HEADERS = [
    "provider",
    "date_of_service",
    "procedure_code",
    "billed_amount",
    "allowed_amount",
    "deductible",
    "copay",
    "coinsurance",
    "ineligible",
    "discount_amount",
    "other_plan_payment",
    "other_adjustments",
    "net_payment_amount"
]


PROMPT = """
ROLE:
You are a highly accurate OCR table extraction model.

Extract ONLY the service table from the image.

Return ONLY valid JSON.

-------------------------
GENERAL RULES
-------------------------

1. Extract every service row exactly as shown.

2. Do NOT skip rows.

3. Do NOT duplicate rows.

4. Preserve the original row order.

5. If a cell is blank, return "".

6. Do NOT infer, calculate, or modify any value.

7. Copy all monetary values exactly as printed.

8. Extract only the procedure code from the "Code or Description" column.

Examples:
D0150
D0220
D0230
D0274

Do NOT extract the description text.

9. Ignore the following:
- Patient information
- Member information
- Claim information
- Addresses
- Description text
- Reason Codes

10. Extract the Totals row separately into "column_totals".

11. Do NOT include the Totals row as a service row.

-------------------------
COLUMN MAPPING
-------------------------
Provider:
-> provider

Service Date(s)
→ date_of_service

Code or Description
→ procedure_code

Billed Amount
→ billed_amount

Allowed Amount
→ allowed_amount

Deductible
→ deductible

Co-Pay
→ copay

Co-Ins
→ coinsurance

Ineligible
→ ineligible

Discount Amount
→ discount_amount

Other Plan Payment
→ other_plan_payment

Other Adjustments
→ other_adjustments

Net Payment Amount
→ net_payment_amount
-------------------------
OUTPUT FORMAT
-------------------------

{
  "rows": [
    {
      "provider": {
        "value": "",
        "confidence": 0.0
      },

      "date_of_service": {
        "value": "",
        "confidence": 0.0
      },

      "procedure_code": {
        "value": "",
        "confidence": 0.0
      },

      "billed_amount": {
        "value": "",
        "confidence": 0.0
      },

      "allowed_amount": {
        "value": "",
        "confidence": 0.0
      },

      "deductible": {
        "value": "",
        "confidence": 0.0
      },

      "copay": {
        "value": "",
        "confidence": 0.0
      },

      "coinsurance": {
        "value": "",
        "confidence": 0.0
      },

      "ineligible": {
        "value": "",
        "confidence": 0.0
      },

      "discount_amount": {
        "value": "",
        "confidence": 0.0
      },

      "other_plan_payment": {
        "value": "",
        "confidence": 0.0
      },

      "other_adjustments": {
        "value": "",
        "confidence": 0.0
      },

      "net_payment_amount": {
        "value": "",
        "confidence": 0.0
      }
    }
  ],

  "column_totals": {
    "billed_amount": {
      "value": "",
      "confidence": 0.0
    },

    "allowed_amount": {
      "value": "",
      "confidence": 0.0
    },

    "deductible": {
      "value": "",
      "confidence": 0.0
    },

    "copay": {
      "value": "",
      "confidence": 0.0
    },

    "coinsurance": {
      "value": "",
      "confidence": 0.0
    },

    "ineligible": {
      "value": "",
      "confidence": 0.0
    },

    "discount_amount": {
      "value": "",
      "confidence": 0.0
    },

    "other_plan_payment": {
      "value": "",
      "confidence": 0.0
    },

    "other_adjustments": {
      "value": "",
      "confidence": 0.0
    },

    "net_payment_amount": {
      "value": "",
      "confidence": 0.0
    }
  }
}

 For every extracted field, return:
   - value
   - confidence
 
VALUE + CONFIDENCE RULES:
 
For every field return:
{
  "value": "",
  "confidence": ""
}
 
VALUE:
- "value" = the exact text/value visibly present in the specified location.
- Read ONLY from the exact cell/row/column requested.
- Copy exactly as printed; preserve "$" and formatting when visible.
- Never guess, infer, calculate, copy, shift, or use values from another row,
  column, table section, or Totals row.
- If the exact location is blank, missing, or has no clearly readable value:
  value = ""
 
CONFIDENCE:
- "confidence" = confidence that the extracted value is actually present
  in that exact location.
- Use a number from 0.0 to 1.0 based ONLY on visual evidence.
- 1.0 = clearly visible and certain.
- 0.8–0.99 = clearly visible with minor uncertainty.
- 0.5–0.79 = visible but difficult/ambiguous.
- 0.1–0.49 = very unclear.
- 0.0 = blank, missing, or no reliable visual evidence.
 
IMPORTANT:
Confidence is NOT confidence that the value is mathematically correct
or logically expected. It is ONLY confidence that the value shown in
"value" is what is visibly printed in the exact requested location.
 
If value = "":
confidence MUST = 0.0.

Return ONLY the JSON.
"""



def enforce_schema(parsed_output):

    allowed_columns = COLUMN_HEADERS  # ✅ ordered list

    for table in parsed_output.get("tables", []):

        # =========================
        # 🔹 CLEAN ROWS (STRICT + ORDERED)
        # =========================
        cleaned_rows = []

        for row in table.get("rows", []):

            cleaned_row = {}

            for col in allowed_columns:
                cleaned_row[col] = row.get(col, "")

            procedure_code = cleaned_row.get("procedure_code", "").strip()

            match = re.search(r"D\d{4}", procedure_code)

            if not match:
                continue

            cleaned_row["procedure_code"] = match.group()

            cleaned_rows.append(cleaned_row)
        table["rows"] = cleaned_rows

        # =========================
        # 🔹 CLEAN COLUMN TOTALS (STRICT + ORDERED)
        # =========================
        totals = table.get("column_totals", {})

        totals.pop("provider", None)
        totals.pop("date_of_service", None)
        totals.pop("procedure_code", None)
        ordered_totals = {}

        for col in allowed_columns:
            if col not in ["date_of_service", "procedure_code", "provider"]:
                ordered_totals[col] = totals.get(col, "")

        table["column_totals"] = ordered_totals

    return parsed_output

def extract_table_from_image(image_path, expected_rows=None):

    image = Image.open(image_path).convert("RGB")

    row_instruction = ""

    if expected_rows is not None:
        row_instruction = f"""
IMPORTANT ROW COUNT:

An independent PDF analysis detected EXACTLY {expected_rows}
physical service row(s) in this image.

You MUST return exactly {expected_rows} item(s) in "rows".

IMPORTANT:
- Count physical service rows, not unique values.
- Two physically separate rows may contain identical values.
- If two physical rows are identical, KEEP BOTH.
- Do NOT merge identical rows.
- Do NOT create additional rows.
- Do NOT treat the Totals row as a service row.
- Do NOT treat the header as a service row.

The number of objects in "rows" MUST be exactly {expected_rows}.
"""

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": image
                },
                {
                    "type": "text",
                    "text": PROMPT + row_instruction
                }
            ]
        }
    ]

    output = pipe(
        text=messages,
        max_new_tokens=5000,
        temperature=0.0,
        repetition_penalty=1.2
    )

    generated_text = output[0]["generated_text"]

    if isinstance(generated_text, list):
        generated_text = generated_text[-1]["content"]

    generated_text = generated_text.strip()

    # REMOVE MARKDOWN IF MODEL RETURNS IT
    generated_text = generated_text.replace(
        "```json",
        ""
    ).replace(
        "```",
        ""
    ).strip()

    return generated_text



def parse_amount(val):
    if val in ["", None]:
        return 0.0
    return float(str(val).replace("$", "").replace(",", "").strip())


LABEL_STOPWORDS = {
    "patient", "participant", "customer", "claim", "group",
    "member", "provider", "administered", "acct", "service",
    "id", "number", "address", "date", "tax"
}

def clean_repeated_word(text):
    """Undo font-duplication (e.g. 'ppppaaaattttiiiieeeennnntttt' -> 'patient')
    while preserving legitimate double letters (e.g. 'matthews')."""
    return re.sub(r'(.)\1{2,}', r'\1', text)


def extract_patient_name(page):
    words = page.extract_words()

    # Normalize every word's text, keep original position data
    norm_words = [
        {**w, "clean": clean_repeated_word(w["text"])}
        for w in words
    ]

    # Find "Patient" followed by "Name..." (case-insensitive, punctuation-stripped)
    patient_label_idx = None
    for i in range(len(norm_words) - 1):
        w1 = norm_words[i]["clean"].strip(":#").lower()
        w2 = norm_words[i + 1]["clean"].strip(":#").lower()
        if w1 == "patient" and w2.startswith("name"):
            patient_label_idx = i + 1
            break

    if patient_label_idx is None:
        return ""

    label_word = norm_words[patient_label_idx]
    label_right = label_word["x1"]
    y = label_word["top"]

    name_parts = []
    for w in norm_words[patient_label_idx + 1:]:
        # stop once we leave the same line
        if abs(w["top"] - y) > 3:
            break
        # skip anything to the left of / under the label itself
        if w["x0"] <= label_right:
            continue

        token = w["clean"].strip()
        token_key = token.strip(":#").lower()

        # Stop at the next label: either a known label word, or a token
        # containing ':' / '#' (glued "Label:" / "Acct####:" patterns)
        if token_key in LABEL_STOPWORDS or ":" in token or "#" in token:
            break

        name_parts.append(token)

    return " ".join(name_parts).title()


def check_claim_denied(pdf_path):

    denial_keywords = [
        "denied",
        "denial"
    ]

    skip_phrase = "important information about your"

    with pdfplumber.open(pdf_path) as pdf:

        for page_num, page in enumerate(pdf.pages, start=1):

            text = page.extract_text() or ""
            text = normalize_repeated_chars(text).lower()
            # print(f"normalized text --------------->>> : {text}")

            # Skip Appeal Rights page
            if skip_phrase in text:
                # print(f"this is the skipping phrase text ---------- : {skip_phrase in text}")
                print(f"⏭️ Skipping page {page_num}")
                continue

            for keyword in denial_keywords:
                if keyword in text:
                    print(f"❌ Claim denied keyword '{keyword}' found on page {page_num}")
                    return "denied"

    return "not denied"

def validate_eob_table(table: dict, table_index: int):

    rows = table.get("rows", [])
    totals = table.get("column_totals", {})

    # field_names = list(compute_totals_from_services([]).keys())   # ADD
    # total_fields = len(field_names) 

    if not rows:
        return False, "", [], 0

    computed_totals = {
        "billed_amount": round(sum(parse_amount(r.get("billed_amount", "")) for r in rows), 2),
        "allowed_amount": round(sum(parse_amount(r.get("allowed_amount", "")) for r in rows), 2),
        "deductible": round(sum(parse_amount(r.get("deductible", "")) for r in rows), 2),
        "copay": round(sum(parse_amount(r.get("copay", "")) for r in rows), 2),
        "coinsurance": round(sum(parse_amount(r.get("coinsurance", "")) for r in rows), 2),
        "ineligible": round(sum(parse_amount(r.get("ineligible", "")) for r in rows), 2),
        "discount_amount": round(sum(parse_amount(r.get("discount_amount", "")) for r in rows), 2),
        "other_plan_payment": round(sum(parse_amount(r.get("other_plan_payment", "")) for r in rows), 2),
        "other_adjustments": round(sum(parse_amount(r.get("other_adjustments", "")) for r in rows), 2),
        "net_payment_amount": round(sum(parse_amount(r.get("net_payment_amount", "")) for r in rows), 2),
    }
    total_fields = len(computed_totals) 

    result_validation = ""
    errors = []
    has_error = False

    print(f"\n🔍 Validation for [Table {table_index}]")
    print("-" * 75)

    for field, computed_value in computed_totals.items():

        extracted_value = round(parse_amount(totals.get(field, "")), 2)

        if computed_value == extracted_value:
            icon = "✅"
            status = "match"
        else:
            icon = "❌"
            status = "MISMATCH"
            has_error = True

            errors.append({
                "field": field,
                "computed": computed_value,
                "extracted": extracted_value
            })

        line = f"{icon} {field:25s} computed={computed_value:<10} | extracted={extracted_value:<10} {status}"
        print(line)
        result_validation += "\n" + line

    if has_error:
        print(f"❌ [Table {table_index}] Validation FAILED\n")
        return False, result_validation, errors, total_fields
    else:
        print(f"✅ [Table {table_index}] Validation PASSED\n")
        return True, result_validation, [], total_fields
    


def count_service_rows(regions):
    """
    regions: list of (page, region_top, region_bottom) tuples.
    For a single-page block, pass one region.
    For a merged (cross-page) block, pass one region per page:
        [(prev_page, start_y, prev_page.height), (curr_page, 0, end_y)]
    """
    total = 0

    for page, region_top, region_bottom in regions:

        words = page.extract_words()
        row_positions = []

        for w in words:
            text = w["text"].strip()

            match = re.search(r"\bD\d{4}\b", text)

            if match:
                y = float(w["top"])

                if region_top <= y <= region_bottom:
                    row_positions.append(y)

        row_positions.sort()

        grouped_rows = []
        threshold = 3

        for y in row_positions:
            if not grouped_rows:
                grouped_rows.append(y)
            else:
                if abs(y - grouped_rows[-1]) > threshold:
                    grouped_rows.append(y)

        total += len(grouped_rows)

    return total


def validate_service_row_count(regions, table, table_index):

    detected_count = count_service_rows(regions)

    rows = table.get("rows", [])
    extracted_count = len([
        r for r in rows
        if r.get("procedure_code") not in ["", None]
    ])

    print(f"\n📊 Row Count Validation [Table {table_index}]")
    print("-" * 70)

    if detected_count == extracted_count:
        icon = "✅"
        status = "match"
    else:
        icon = "❌"
        status = "MISMATCH"

    print(f"{icon} row_count detected={detected_count:<5} | extracted={extracted_count:<5} {status}")
    print("-" * 70)

    return detected_count == extracted_count



def merge_images_vertically(img1, img2):
    """
    Stack two PIL images vertically (img1 on top, img2 below),
    aligning on width (using the wider of the two as canvas width).
    """
    width = max(img1.width, img2.width)
    total_height = img1.height + img2.height

    merged = Image.new("RGB", (width, total_height), "white")
    merged.paste(img1, (0, 0))
    merged.paste(img2, (0, img1.height))

    return merged


# def normalize_repeated_chars(text):
#     """
#     PPPPaaaattttiiiieeeennnntttt -> Patient
#     """
#     if text is None:
#         return ""
#     return re.sub(r'(.)\1{2,}', r'\1', text)

def normalize_repeated_chars(text):
    return re.sub(r'(.)\1+', r'\1', text)


def run_pipeline(
    pdf_path,
    output_dir="EOB_OUTPUT/ACS_Health",
    start_anchor="Patient Name:",
    end_anchor="Totals:", company_name = "ACS health"
):
    """
    Full pipeline (rotation removed):
        1. Open PDF directly.
        2. Crop each Claim block using event-based start/end anchor pairing,
           merging blocks that split across pages.
        3. Save images.
        4. Run VLM extraction on each crop.
        5. Validate + build final JSON.
    """

    pdf_name = os.path.splitext(os.path.basename(pdf_path))[0]
    pdf_full_name = os.path.basename(pdf_path)



    working_dir = os.path.join(output_dir, pdf_name)
    os.makedirs(working_dir, exist_ok=True)

    claim_status = check_claim_denied(pdf_path)
    print(f"Claim Status: {claim_status}")


    results = []
    all_patients = []
    block_counter = 0

    # Holds an orphan (unclosed) crop carried over from the previous page
    # pending = {"image": PIL.Image, "page_num": int, "start_top": float}
    pending = None

    with pdfplumber.open(pdf_path) as pdf:

        for page_idx, page in enumerate(pdf.pages):

            print(f"\nScanning Page {page_idx + 1}")
            # print(page.extract_text())
            

            # print("Search result:", page.search(skip_phrase))

            print(page.search("Important Information About Your"))

            lines = page.extract_text_lines()

            events = []
            for line in lines:
                text = normalize_repeated_chars(line["text"])
                # print(text[:300] if text else "No text")

                if start_anchor in text:
                    events.append(("start", line["top"]))

                if end_anchor in text:
                    events.append(("end", line["bottom"] + 10))

            events.sort(key=lambda e: e[1])

            cursor_start_y = None

            for kind, y in events:

                if kind == "start":
                    cursor_start_y = y
                    continue

                # kind == "end"
                output_path = None


                if pending is not None:
                    # Closes an orphan block that started on a previous page.
                    bbox = (0, 0, page.width, y)
                    cropped_page = page.crop(bbox)
                    continuation_img = cropped_page.to_image(resolution=300).original

                    merged_img = merge_images_vertically(
                        pending["image"], continuation_img
                    )

                    block_counter += 1
                    output_path = os.path.join(
                        working_dir,
                        f"block_{block_counter:03d}_merged_"
                        f"p{pending['page_num']+1}_p{page_idx+1}.png"
                    )
                    merged_img.save(output_path)
                    print(f"Saved (merged) -> {output_path}")

                    results.append({
                        "page": f"{pending['page_num']+1}-{page_idx+1}",
                        "block": block_counter,
                        "file": output_path
                    })

                    regions = [
                        (pending["page"], pending["start_top"], pending["page"].height),
                        (page, 0, y)
                    ]

                    pending = None
                    cursor_start_y = None

                elif cursor_start_y is not None:

                    patient_name = extract_patient_name(page)
                    print(f" Patient Name: {patient_name}")
                    # Normal, single-page complete block.

                    top_padding = 25
                    top = max(0, cursor_start_y - top_padding)

                    bbox = (0, top, page.width, y)
                    cropped_page = page.crop(bbox)

                    block_counter += 1
                    output_path = os.path.join(
                        working_dir,
                        f"block_{block_counter:03d}_p{page_idx+1}.png"
                    )
                    cropped_page.to_image(resolution=300).save(output_path)
                    print(f"Saved -> {output_path}")

                    results.append({
                        "page": page_idx + 1,
                        "block": block_counter,
                        "file": output_path
                    })
                    regions = [
                            (page, cursor_start_y, y)
                        ]

                    cursor_start_y = None

                else:
                    # Stray end anchor with no matching start - ignore
                    continue

                # =====================================
                # VLM EXTRACTION + VALIDATION
                # =====================================
                try:
                    expected_rows = count_service_rows(regions)
                    print(f"📌 Expected physical service rows: {expected_rows}")
                    llm_output = extract_table_from_image(
                                output_path,
                                expected_rows=expected_rows
                            )
                    print("=" * 100)
                    print(llm_output)
                    print("=" * 100)
                    parsed_output = json.loads(llm_output)

                    model_confidence = calculate_model_confidence(parsed_output)   # ADD
                    parsed_output = _unwrap_vlm_output(parsed_output)


                    parsed_output = {"tables": [parsed_output]}
                    parsed_output = enforce_schema(parsed_output)

                    # =====================================
                    # 🔹 ENFORCE EXPECTED ROW COUNT
                    # =====================================
                    for table in parsed_output.get("tables", []):

                        rows = table.get("rows", [])

                        if expected_rows is not None and len(rows) > expected_rows:

                            print(
                                f"⚠️ VLM returned {len(rows)} rows, "
                                f"but PDF detected {expected_rows} rows."
                            )

                            # IMPORTANT:
                            # Do NOT remove duplicate rows based on their values.
                            # If two physical rows are identical, both must be preserved.
                            rows = rows[:expected_rows]

                            table["rows"] = rows


                    patient_name = extract_patient_name(page)
                    print(f" Patient Name: {patient_name}")

                    for table in parsed_output.get("tables", []):
                        new_table = {
                            "EOB_ID": pdf_name,
                            "patient_name": patient_name,
                            "rows": table.get("rows", []),
                            "column_totals": table.get("column_totals", {}),
                            "_model_confidence": model_confidence, 
                        }
                        table.clear()
                        table.update(new_table)

                    for t_idx, table in enumerate(parsed_output.get("tables", []), start=1):
                        row_count_ok = validate_service_row_count(
                            regions,
                            table,
                            t_idx
                        )

                    # expected_rows = count_service_rows(regions)

                    for t_idx, table in enumerate(parsed_output.get("tables", []), start=1):

                        is_valid, log, errors, total_fields  = validate_eob_table(table, t_idx)
                        if not row_count_ok:
                            is_valid = False
                            errors = errors + [{"type": "row_count_mismatch"}]


                    for table in parsed_output.get("tables", []):
                        structured_table = {
                            "EOB_ID": pdf_name,
                            "patient_name": table.get("patient_name", ""),
                            "rows": table.get("rows", []),
                            "totals": table.get("column_totals", {}),
                            "validation": {"status": is_valid, "errors": errors},
                            "_expected_rows": expected_rows,                          # ADD
                            "_total_fields": total_fields,                             # ADD
                            "_model_confidence": table.get("_model_confidence", 0.0), 
                        }
                        print(f"✅ Completed Block {block_counter} on Page {page_idx + 1}")

                    date_of_service = ""
                    provider = ""
                    if structured_table.get("rows"):
                        date_of_service = structured_table["rows"][0].get("date_of_service", "")
                        provider = structured_table["rows"][0].get("provider", "")

                    for row in structured_table.get("rows", []):
                        for col in ["service_description", "date_of_service", "provider"]:
                            row.pop(col, None)

                    patient_data = {
                        "patient_name": structured_table.get("patient_name", ""),
                        "date_of_service": date_of_service,
                        "provider": provider,
                        "services": structured_table.get("rows", []),
                        "totals": structured_table.get("totals", {}),
                        "validation": structured_table.get("validation"),
                        "_expected_rows": structured_table.get("_expected_rows", 0),               # ADD
                        "_total_fields": structured_table.get("_total_fields", 0),                  # ADD
                        "_model_confidence": structured_table.get("_model_confidence", 0.0),
                    }

                    all_patients.append(patient_data)

                except Exception as e:
                    print(f"Error: {e}")

            # Leftover unmatched start at end of page -> orphan, carry forward
            if cursor_start_y is not None:
                top_padding = 30
                top = max(0, cursor_start_y - top_padding)
                bbox = (0, top, page.width, page.height)
                cropped_page = page.crop(bbox)
                orphan_img = cropped_page.to_image(resolution=300).original

                pending = {
                    "image": orphan_img,
                    "page_num": page_idx,
                    "page": page,
                    "start_top": cursor_start_y
                }
                print(
                    f"Orphan block detected on page {page_idx + 1} "
                    f"(no '{end_anchor}' found) -> carrying forward"
                )

    print("\n===================================")
    print(f"Total Crops Created : {len(results)}")
    print("===================================")

    confidence_score = calculate_eob_confidence(all_patients)   # ADD

    for patient in all_patients:           
                             # ADD cleanup
        patient.pop("_expected_rows", None)
        patient.pop("_total_fields", None)
        patient.pop("_model_confidence", None)

    final = [
        {
            "eob_id": pdf_name,
            "file_name":pdf_full_name,
            "claim_status": claim_status,
            "payor": "Dental Blue",
            "confidence_score": confidence_score,
            "patients": all_patients
        }
    ]

    success_path, failed_path = save_split_output(
            final,
            company_name=company_name,
            pdf_name=pdf_name,
            pdf_path=pdf_path,
            cropped_dir=working_dir,
        )
    
    print(f"\n📁 Cropped images : {working_dir}")
    print(f"✅ Success json   : {success_path}")
    print(f"⚠  Failed json    : {failed_path}")
    return final

W0901 17:18:05.424000 3285280 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0901 17:18:05.439000 3285280 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

## Test 1

In [2]:
import os

folder_path = r"/home/cipl/users/OCR_Project/ACS_health/pdf"

for filename in os.listdir(folder_path):
    if filename.lower().endswith(".pdf"):
        pdf_path = os.path.join(folder_path, filename)

        print(f"\n{'='*80}")
        print(f"Processing: {filename}")
        print(f"{'='*80}")

        try:
            run_pipeline(pdf_path)
            print(f"✅ Completed: {filename}")

        except Exception as e:
            print(f"❌ Failed: {filename}")
            print(f"Error: {e}")


Processing: Pmt_EOP_889084725.pdf
⏭️ Skipping page 2
⏭️ Skipping page 4
Claim Status: not denied

Scanning Page 1
[]
 Patient Name: Leanne Burnett


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Keyword argument `temperature` is not a valid argument for this processor and will be ignored.
[transformers] Keyword argument `repetition_penalty` is not a valid argument for this processor and will be ignored.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=5000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Saved -> EOB_OUTPUT/ACS_Health/Pmt_EOP_889084725/block_001_p1.png


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Both `max_new_tokens` (=5000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{
  "rows": [
    {
      "provider": {
        "value": "DUC TANG",
        "confidence": 1.0
      },
      "date_of_service": {
        "value": "05/12/26",
        "confidence": 1.0
      },
      "procedure_code": {
        "value": "D0120-PERIODIC ORAL EVALUATION - EST PATIENT",
        "confidence": 1.0
      },
      "billed_amount": {
        "value": "95.00",
        "confidence": 1.0
      },
      "allowed_amount": {
        "value": "53.00",
        "confidence": 1.0
      },
      "deductible": {
        "value": "0.00",
        "confidence": 1.0
      },
      "copay": {
        "value": "0.00",
        "confidence": 1.0
      },
      "coinsurance": {
        "value": "0.00",
        "confidence": 1.0
      },
      "ineligible": {
        "value": "0.00",
        "confidence": 1.0
      },
      "discount_amount": {
        "value": "42.00",
        "confidence": 1.0
      },
      "other_plan_payment": {
        "value": "0.00",
        "confidence": 1.0
      },
    

...

Saved -> EOB_OUTPUT/ACS_Health/Pmt_EOP_896761030/block_008_p7.png


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Both `max_new_tokens` (=5000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{
  "rows": [
    {
      "provider": {
        "value": "DUC TANG",
        "confidence": 1.0
      },
      "date_of_service": {
        "value": "05/15/26",
        "confidence": 1.0
      },
      "procedure_code": {
        "value": "D5650-ADD TOOTH TO EXISTING PARTIAL DENTURE",
        "confidence": 1.0
      },
      "billed_amount": {
        "value": "337.00",
        "confidence": 1.0
      },
      "allowed_amount": {
        "value": "204.00",
        "confidence": 1.0
      },
      "deductible": {
        "value": "50.00",
        "confidence": 1.0
      },
      "copay": {
        "value": "0.00",
        "confidence": 1.0
      },
      "coinsurance": {
        "value": "77.00",
        "confidence": 1.0
      },
      "ineligible": {
        "value": "0.00",
        "confidence": 1.0
      },
      "discount_amount": {
        "value": "133.00",
        "confidence": 1.0
      },
      "other_plan_payment": {
        "value": "0.00",
        "confidence": 1.0
      },


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Both `max_new_tokens` (=5000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{
  "rows": [
    {
      "provider": {
        "value": "DUC TANG",
        "confidence": 1.0
      },
      "date_of_service": {
        "value": "05/18/26",
        "confidence": 1.0
      },
      "procedure_code": {
        "value": "D2644-ONLAY - PORCELAIN/CERAMIC - 4 OR MORE SURFACES",
        "confidence": 1.0
      },
      "billed_amount": {
        "value": "1,933.00",
        "confidence": 1.0
      },
      "allowed_amount": {
        "value": "0.00",
        "confidence": 1.0
      },
      "deductible": {
        "value": "0.00",
        "confidence": 1.0
      },
      "copay": {
        "value": "0.00",
        "confidence": 1.0
      },
      "coinsurance": {
        "value": "0.00",
        "confidence": 1.0
      },
      "ineligible": {
        "value": "1,327.00",
        "confidence": 1.0
      },
      "discount_amount": {
        "value": "606.00",
        "confidence": 1.0
      },
      "other_plan_payment": {
        "value": "0.00",
        "confidence": 1.

## test 2

In [2]:
import os

folder_path = r"/home/cipl/users/OCR_Project/ACS_health/pdf"

for filename in os.listdir(folder_path):
    if filename.lower().endswith(".pdf"):
        pdf_path = os.path.join(folder_path, filename)

        print(f"\n{'='*80}")
        print(f"Processing: {filename}")
        print(f"{'='*80}")

        try:
            run_pipeline(pdf_path)
            print(f"✅ Completed: {filename}")

        except Exception as e:
            print(f"❌ Failed: {filename}")
            print(f"Error: {e}")


Processing: Pmt_EOP_889084725.pdf
⏭️ Skipping page 2
⏭️ Skipping page 4
Claim Status: not denied

Scanning Page 1


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Keyword argument `temperature` is not a valid argument for this processor and will be ignored.
[transformers] Keyword argument `repetition_penalty` is not a valid argument for this processor and will be ignored.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


[]
 Patient Name: Leanne Burnett
Saved -> EOB_OUTPUT/ACS_Health/Pmt_EOP_889084725/block_001_p1.png
📌 Expected physical service rows: 3


[transformers] Both `max_new_tokens` (=5000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


{
  "rows": [
    {
      "provider": {
        "value": "DUC TANG",
        "confidence": 1.0
      },
      "date_of_service": {
        "value": "05/12/26",
        "confidence": 1.0
      },
      "procedure_code": {
        "value": "D0120-PERIODIC ORAL EVALUATION - EST PATIENT",
        "confidence": 1.0
      },
      "billed_amount": {
        "value": "95.00",
        "confidence": 1.0
      },
      "allowed_amount": {
        "value": "53.00",
        "confidence": 1.0
      },
      "deductible": {
        "value": "0.00",
        "confidence": 1.0
      },
      "copay": {
        "value": "0.00",
        "confidence": 1.0
      },
      "coinsurance": {
        "value": "0.00",
        "confidence": 1.0
      },
      "ineligible": {
        "value": "0.00",
        "confidence": 1.0
      },
      "discount_amount": {
        "value": "42.00",
        "confidence": 1.0
      },
      "other_plan_payment": {
        "value": "0.00",
        "confidence": 1.0
      },
    

...

{
  "rows": [
    {
      "provider": {
        "value": "DUC TANG DDS",
        "confidence": 1.0
      },
      "date_of_service": {
        "value": "03/13/26",
        "confidence": 1.0
      },
      "procedure_code": {
        "value": "D2643-ONLAY - PORCELAIN/CERAMIC - THREE SURFACES",
        "confidence": 1.0
      },
      "billed_amount": {
        "value": "1,848.00",
        "confidence": 1.0
      },
      "allowed_amount": {
        "value": "1,307.00",
        "confidence": 1.0
      },
      "deductible": {
        "value": "75.00",
        "confidence": 1.0
      },
      "copay": {
        "value": "0.00",
        "confidence": 1.0
      },
      "coinsurance": {
        "value": "616.00",
        "confidence": 1.0
      },
      "ineligible": {
        "value": "0.00",
        "confidence": 1.0
      },
      "discount_amount": {
        "value": "541.00",
        "confidence": 1.0
      },
      "other_plan_payment": {
        "value": "0.00",
        "confidence":

[transformers] Both `max_new_tokens` (=5000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Both `max_new_tokens` (=5000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{
  "rows": [
    {
      "provider": {
        "value": "DUC TANG",
        "confidence": 1.0
      },
      "date_of_service": {
        "value": "05/15/26",
        "confidence": 1.0
      },
      "procedure_code": {
        "value": "D5650-ADD TOOTH TO EXISTING PARTIAL DENTURE",
        "confidence": 1.0
      },
      "billed_amount": {
        "value": "337.00",
        "confidence": 1.0
      },
      "allowed_amount": {
        "value": "204.00",
        "confidence": 1.0
      },
      "deductible": {
        "value": "50.00",
        "confidence": 1.0
      },
      "copay": {
        "value": "0.00",
        "confidence": 1.0
      },
      "coinsurance": {
        "value": "77.00",
        "confidence": 1.0
      },
      "ineligible": {
        "value": "0.00",
        "confidence": 1.0
      },
      "discount_amount": {
        "value": "133.00",
        "confidence": 1.0
      },
      "other_plan_payment": {
        "value": "0.00",
        "confidence": 1.0
      },


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Both `max_new_tokens` (=5000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{
  "rows": [
    {
      "provider": {
        "value": "DUC TANG",
        "confidence": 1.0
      },
      "date_of_service": {
        "value": "05/18/26",
        "confidence": 1.0
      },
      "procedure_code": {
        "value": "D2644-ONLAY- PORCELAIN/CERAMIC -4 OR MORE SURFACES",
        "confidence": 1.0
      },
      "billed_amount": {
        "value": "1,933.00",
        "confidence": 1.0
      },
      "allowed_amount": {
        "value": "0.00",
        "confidence": 1.0
      },
      "deductible": {
        "value": "0.00",
        "confidence": 1.0
      },
      "copay": {
        "value": "0.00",
        "confidence": 1.0
      },
      "coinsurance": {
        "value": "0.00",
        "confidence": 1.0
      },
      "ineligible": {
        "value": "1,327.00",
        "confidence": 1.0
      },
      "discount_amount": {
        "value": "606.00",
        "confidence": 1.0
      },
      "other_plan_payment": {
        "value": "0.00",
        "confidence": 1.0


In [2]:
run_pipeline(r"/home/cipl/users/OCR_Project/ACS_health/pdf/Pmt_EOP_896761030.pdf")

⏭️ Skipping page 2
⏭️ Skipping page 6
⏭️ Skipping page 8
Claim Status: not denied

Scanning Page 1
[]
 Patient Name: Jimmy L Webber
Saved -> EOB_OUTPUT/ACS_Health/Pmt_EOP_896761030/block_001_p1.png
📌 Expected physical service rows: 4


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Keyword argument `temperature` is not a valid argument for this processor and will be ignored.
[transformers] Keyword argument `repetition_penalty` is not a valid argument for this processor and will be ignored.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=5000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{
  "rows": [
    {
      "provider": {
        "value": "DUC TANG",
        "confidence": 1.0
      },
      "date_of_service": {
        "value": "01/19/26",
        "confidence": 1.0
      },
      "procedure_code": {
        "value": "D0150-COMP ORAL EVALUATION - NEW OR EST PATIENT",
        "confidence": 1.0
      },
      "billed_amount": {
        "value": "139.00",
        "confidence": 1.0
      },
      "allowed_amount": {
        "value": "87.00",
        "confidence": 1.0
      },
      "deductible": {
        "value": "0.00",
        "confidence": 1.0
      },
      "copay": {
        "value": "0.00",
        "confidence": 1.0
      },
      "coinsurance": {
        "value": "0.00",
        "confidence": 1.0
      },
      "ineligible": {
        "value": "0.00",
        "confidence": 1.0
      },
      "discount_amount": {
        "value": "52.00",
        "confidence": 1.0
      },
      "other_plan_payment": {
        "value": "0.00",
        "confidence": 1.0
      },


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Both `max_new_tokens` (=5000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[]
 Patient Name: Paul Matthews
Saved -> EOB_OUTPUT/ACS_Health/Pmt_EOP_896761030/block_002_p3.png
📌 Expected physical service rows: 2


...

[transformers] Both `max_new_tokens` (=5000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Both `max_new_tokens` (=5000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{
  "rows": [
    {
      "provider": {
        "value": "DUC TANG",
        "confidence": 1.0
      },
      "date_of_service": {
        "value": "05/15/26",
        "confidence": 1.0
      },
      "procedure_code": {
        "value": "D5650-ADD TOOTH TO EXISTING PARTIAL DENTURE",
        "confidence": 1.0
      },
      "billed_amount": {
        "value": "337.00",
        "confidence": 1.0
      },
      "allowed_amount": {
        "value": "204.00",
        "confidence": 1.0
      },
      "deductible": {
        "value": "50.00",
        "confidence": 1.0
      },
      "copay": {
        "value": "0.00",
        "confidence": 1.0
      },
      "coinsurance": {
        "value": "77.00",
        "confidence": 1.0
      },
      "ineligible": {
        "value": "0.00",
        "confidence": 1.0
      },
      "discount_amount": {
        "value": "133.00",
        "confidence": 1.0
      },
      "other_plan_payment": {
        "value": "0.00",
        "confidence": 1.0
      },


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Both `max_new_tokens` (=5000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{
  "rows": [
    {
      "provider": {
        "value": "DUC TANG",
        "confidence": 1.0
      },
      "date_of_service": {
        "value": "05/18/26",
        "confidence": 1.0
      },
      "procedure_code": {
        "value": "D2644-ONLAY - PORCELAIN/CERAMIC - 4 OR MORE SURFACES",
        "confidence": 1.0
      },
      "billed_amount": {
        "value": "1,933.00",
        "confidence": 1.0
      },
      "allowed_amount": {
        "value": "0.00",
        "confidence": 1.0
      },
      "deductible": {
        "value": "0.00",
        "confidence": 1.0
      },
      "copay": {
        "value": "0.00",
        "confidence": 1.0
      },
      "coinsurance": {
        "value": "0.00",
        "confidence": 1.0
      },
      "ineligible": {
        "value": "1,327.00",
        "confidence": 1.0
      },
      "discount_amount": {
        "value": "606.00",
        "confidence": 1.0
      },
      "other_plan_payment": {
        "value": "0.00",
        "confidence": 1.

[{'eob_id': 'Pmt_EOP_896761030',
  'file_name': 'Pmt_EOP_896761030.pdf',
  'claim_status': 'not denied',
  'payor': 'Dental Blue',
  'confidence_score': 100.0,
  'patients': [{'patient_name': 'Jimmy L Webber',
    'date_of_service': '01/19/26',
    'provider': 'DUC TANG',
    'services': [{'procedure_code': 'D0150',
      'billed_amount': '139.00',
      'allowed_amount': '87.00',
      'deductible': '0.00',
      'copay': '0.00',
      'coinsurance': '0.00',
      'ineligible': '0.00',
      'discount_amount': '52.00',
      'other_plan_payment': '0.00',
      'other_adjustments': '0.00',
      'net_payment_amount': '87.00'},
     {'procedure_code': 'D0220',
      'billed_amount': '45.00',
      'allowed_amount': '30.00',
      'deductible': '0.00',
      'copay': '0.00',
      'coinsurance': '0.00',
      'ineligible': '0.00',
      'discount_amount': '15.00',
      'other_plan_payment': '0.00',
      'other_adjustments': '0.00',
      'net_payment_amount': '30.00'},
     {'procedure

# Final

In [2]:
run_pipeline(r"/home/cipl/users/OCR_Project/ACS_health/pdf/Pmt_EOP_896761030.pdf")

⏭️ Skipping page 2
⏭️ Skipping page 6
⏭️ Skipping page 8
Claim Status: not denied

Scanning Page 1


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Keyword argument `temperature` is not a valid argument for this processor and will be ignored.
[transformers] Keyword argument `repetition_penalty` is not a valid argument for this processor and will be ignored.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


[]
 Patient Name: Jimmy L Webber
Saved -> EOB_OUTPUT/ACS_Health/Pmt_EOP_896761030/block_001_p1.png
📌 Expected physical service rows: 4


[transformers] Both `max_new_tokens` (=5000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{
  "rows": [
    {
      "provider": {
        "value": "DUC TANG",
        "confidence": 1.0
      },
      "date_of_service": {
        "value": "01/19/26",
        "confidence": 1.0
      },
      "procedure_code": {
        "value": "D0150-COMP ORAL EVALUATION - NEW OR EST PATIENT",
        "confidence": 1.0
      },
      "billed_amount": {
        "value": "139.00",
        "confidence": 1.0
      },
      "allowed_amount": {
        "value": "87.00",
        "confidence": 1.0
      },
      "deductible": {
        "value": "0.00",
        "confidence": 1.0
      },
      "copay": {
        "value": "0.00",
        "confidence": 1.0
      },
      "coinsurance": {
        "value": "0.00",
        "confidence": 1.0
      },
      "ineligible": {
        "value": "0.00",
        "confidence": 1.0
      },
      "discount_amount": {
        "value": "52.00",
        "confidence": 1.0
      },
      "other_plan_payment": {
        "value": "0.00",
        "confidence": 1.0
      },


...

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Both `max_new_tokens` (=5000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{
  "rows": [
    {
      "provider": {
        "value": "DUC TANG",
        "confidence": 1.0
      },
      "date_of_service": {
        "value": "05/15/26",
        "confidence": 1.0
      },
      "procedure_code": {
        "value": "D5650-ADD TOOTH TO EXISTING PARTIAL DENTURE",
        "confidence": 1.0
      },
      "billed_amount": {
        "value": "337.00",
        "confidence": 1.0
      },
      "allowed_amount": {
        "value": "204.00",
        "confidence": 1.0
      },
      "deductible": {
        "value": "50.00",
        "confidence": 1.0
      },
      "copay": {
        "value": "0.00",
        "confidence": 1.0
      },
      "coinsurance": {
        "value": "77.00",
        "confidence": 1.0
      },
      "ineligible": {
        "value": "0.00",
        "confidence": 1.0
      },
      "discount_amount": {
        "value": "133.00",
        "confidence": 1.0
      },
      "other_plan_payment": {
        "value": "0.00",
        "confidence": 1.0
      },


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Both `max_new_tokens` (=5000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{
  "rows": [
    {
      "provider": {
        "value": "DUC TANG",
        "confidence": 1.0
      },
      "date_of_service": {
        "value": "05/18/26",
        "confidence": 1.0
      },
      "procedure_code": {
        "value": "D2644-ONLAY- PORCELAN/CERAMIC - 4 OR MORE SURFACES",
        "confidence": 1.0
      },
      "billed_amount": {
        "value": "1,933.00",
        "confidence": 1.0
      },
      "allowed_amount": {
        "value": "0.00",
        "confidence": 1.0
      },
      "deductible": {
        "value": "0.00",
        "confidence": 1.0
      },
      "copay": {
        "value": "0.00",
        "confidence": 1.0
      },
      "coinsurance": {
        "value": "0.00",
        "confidence": 1.0
      },
      "ineligible": {
        "value": "1,327.00",
        "confidence": 1.0
      },
      "discount_amount": {
        "value": "606.00",
        "confidence": 1.0
      },
      "other_plan_payment": {
        "value": "0.00",
        "confidence": 1.0


[{'eob_id': 'Pmt_EOP_896761030',
  'file_name': 'Pmt_EOP_896761030.pdf',
  'claim_status': 'not denied',
  'payor': 'Dental Blue',
  'confidence_score': 99.9,
  'patients': [{'patient_name': 'Jimmy L Webber',
    'date_of_service': '01/19/26',
    'provider': 'DUC TANG',
    'services': [{'procedure_code': 'D0150',
      'billed_amount': '139.00',
      'allowed_amount': '87.00',
      'deductible': '0.00',
      'copay': '0.00',
      'coinsurance': '0.00',
      'ineligible': '0.00',
      'discount_amount': '52.00',
      'other_plan_payment': '0.00',
      'other_adjustments': '0.00',
      'net_payment_amount': '87.00'},
     {'procedure_code': 'D0220',
      'billed_amount': '45.00',
      'allowed_amount': '30.00',
      'deductible': '0.00',
      'copay': '0.00',
      'coinsurance': '0.00',
      'ineligible': '0.00',
      'discount_amount': '15.00',
      'other_plan_payment': '0.00',
      'other_adjustments': '0.00',
      'net_payment_amount': '30.00'},
     {'procedure_